# AutoSelect 教程（更强调上手）

这个版本聚焦 3 件事：
1. 用 `auto_normalize` / `auto_impute` 快速决策
2. 用 `AutoSelector` 一次跑多阶段
3. 导出报告用于复盘和团队沟通

## 0. 导入与数据准备

In [ ]:
from pathlib import Path

import numpy as np
import polars as pl

import scptensor.autoselect as auto
from scptensor.io import load_diann

In [ ]:
def locate_project_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "dia" / "diann").exists():
            return candidate
    raise FileNotFoundError("Cannot locate project root containing data/dia/diann")


def load_diann_subset(
    report_path: Path,
    assay_name: str = "proteins",
    n_samples: int = 90,
    n_features: int = 180,
):
    container = load_diann(
        report_path,
        level="protein",
        table_format="long",
        assay_name=assay_name,
    )
    assay = container.assays[assay_name]
    layer = assay.layers["raw"]

    keep_samples = min(n_samples, container.n_samples)
    keep_features = min(n_features, assay.n_features)

    layer.X = layer.X[:keep_samples, :keep_features]
    if layer.M is not None:
        layer.M = layer.M[:keep_samples, :keep_features]

    container.obs = container.obs.slice(0, keep_samples)
    assay.var = assay.var.slice(0, keep_features)
    return container


ROOT = locate_project_root()
DIANN_REPORT = ROOT / "data" / "dia" / "diann" / "PXD054343" / "1_SC_LF_report.tsv"
container = load_diann_subset(DIANN_REPORT)

print(
    f"Subset shape: {container.n_samples} samples x {container.assays['proteins'].n_features} features"
)
print(f"Missing rate: {np.isnan(container.assays['proteins'].layers['raw'].X).mean():.2%}")

## 1. 单阶段：自动选标准化

In [ ]:
container_norm, norm_report = auto.auto_normalize(
    container=container,
    assay_name="proteins",
    source_layer="raw",
    keep_all=True,
)

print(f"Best normalization method: {norm_report.best_method}")
print(f"Result layer: {norm_report.best_result.layer_name if norm_report.best_result else 'N/A'}")

norm_df = pl.DataFrame(
    [
        {
            "method": r.method_name,
            "overall_score": r.overall_score,
            "exec_time_s": r.execution_time,
            "error": r.error,
        }
        for r in norm_report.results
    ]
).sort("overall_score", descending=True)

norm_df

## 2. 单阶段：在最佳标准化层上自动选填充

In [ ]:
norm_layer = norm_report.best_result.layer_name if norm_report.best_result else "raw"
before_missing = np.isnan(container_norm.assays["proteins"].layers[norm_layer].X).mean()

container_imp, impute_report = auto.auto_impute(
    container=container_norm,
    assay_name="proteins",
    source_layer=norm_layer,
    keep_all=False,
)

imp_layer = impute_report.best_result.layer_name if impute_report.best_result else norm_layer
after_missing = np.isnan(container_imp.assays["proteins"].layers[imp_layer].X).mean()

print(f"Best imputation method: {impute_report.best_method}")
print(f"Input layer:  {norm_layer}")
print(f"Output layer: {imp_layer}")
print(f"Missing rate: {before_missing:.2%} -> {after_missing:.2%}")

## 3. 多阶段：一次完成 normalize -> impute -> reduce -> cluster

In [ ]:
selector = auto.AutoSelector(
    stages=["normalize", "impute", "reduce", "cluster"],
    keep_all=False,
    parallel=False,
    n_jobs=1,
)

result_container, full_report = selector.run(
    container=container,
    assay_name="proteins",
    initial_layer="raw",
)

print(full_report.summary())

for stage_name, stage_report in full_report.stages.items():
    best_layer = stage_report.best_result.layer_name if stage_report.best_result else "N/A"
    print(f"{stage_name:14s} | best={stage_report.best_method:12s} | layer={best_layer}")

## 4. 自定义权重（把选择逻辑改成你的业务偏好）

In [ ]:
weighted_selector = auto.AutoSelector(
    stages=["normalize"],
    keep_all=True,
    weights={
        "normalize": {
            "intragroup_variation": 0.45,
            "intergroup_preservation": 0.35,
            "technical_variance": 0.10,
            "clustering_quality": 0.10,
        }
    },
    parallel=False,
)

_, weighted_report = weighted_selector.run(
    container=container,
    assay_name="proteins",
    initial_layer="raw",
)

print(f"Weighted best normalization method: {weighted_report.stages['normalization'].best_method}")

## 5. 保存报告

In [ ]:
report_dir = ROOT / "docs" / "_tutorial_outputs" / "autoselect"
report_dir.mkdir(parents=True, exist_ok=True)

md_path = report_dir / "autoselect_report.md"
json_path = report_dir / "autoselect_report.json"
csv_path = report_dir / "autoselect_report.csv"

full_report.save(md_path, format="markdown")
full_report.save(json_path, format="json")
full_report.save(csv_path, format="csv")

print(f"Saved: {md_path}")
print(f"Saved: {json_path}")
print(f"Saved: {csv_path}")

## 小结

推荐实战用法：
- 先用 `auto_normalize` / `auto_impute` 快速拿到默认最优解。
- 再用 `AutoSelector` 跑完整流程并固定配置。
- 最后把报告入库，作为参数变更和版本迭代的依据。